##Phase 3: The Bronze Layer – Raw Ingestion

#####Step 3.1: Create Notebook
#####Step 3.2: Ingest Batch Files via Auto Loader

In [0]:
df_customers = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.inferColumnTypes", "true")
        .option(
            "cloudFiles.schemaLocation",
            "/Volumes/dd_hr_sandpit/ecommerce_project/landing_zone/schema/customers"
        )
        .load("/Volumes/dd_hr_sandpit/ecommerce_project/landing_zone/batch/customers/")
)

(
    df_customers.writeStream
        .format("delta")
        .option(
            "checkpointLocation",
            "/Volumes/dd_hr_sandpit/ecommerce_project/landing_zone/checkpoints/bronze_customers"
        )
        .trigger(availableNow=True)
        .toTable("dd_hr_sandpit.ecommerce_project.bronze_customers")
)

In [0]:

df_products = (spark.readStream
                .format("cloudFiles")
                .option("cloudFiles.format", "csv")
                .option("cloudFiles.inferColumnTypes", "true")
                .option(
                    "cloudFiles.schemaLocation",
                    "/Volumes/dd_hr_sandpit/ecommerce_project/landing_zone/schema/products"
                )
                .load("/Volumes/dd_hr_sandpit/ecommerce_project/landing_zone/batch/products/"))

(df_products.writeStream
 .format("delta")
 .option("checkpointLocation", "/Volumes/dd_hr_sandpit/ecommerce_project/landing_zone/checkpoints/bronze_products")
 .trigger(availableNow=True) # Run as a batch micro-batch
 .toTable("dd_hr_sandpit.ecommerce_project.bronze_products"))

##Step 3.4: Repeat for Clickstream. 
Do the exact same thing for the streaming JSON data, saving it to dev_catalog.ecommerce_project.bronze_clickstream. Keep the JSON as a raw string format for now.

In [0]:
# 1. Read clickstream JSON files from the streaming Volume
# We do not use schema inference here to keep the JSON content raw as a string/variant
df_clickstream = (spark.readStream
                  .format("cloudFiles")
                  .option("cloudFiles.format", "json")
                  .option("cloudFiles.schemaLocation", "/Volumes/dd_hr_sandpit/ecommerce_project/landing_zone/streaming/checkpoints/clickstream_schema")
                  .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
                  .load("/Volumes/dd_hr_sandpit/ecommerce_project/landing_zone/streaming/clickstream_json/"))

# 2. Stream continuously into the Bronze Delta table
# Note: For an active continuous stream, we remove trigger(availableNow=True) so it runs non-stop
query_clickstream = (df_clickstream.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/dd_hr_sandpit/ecommerce_project/landing_zone/streaming/checkpoints/bronze_clickstream")
    .trigger(availableNow=True).toTable("dd_hr_sandpit.ecommerce_project.bronze_clickstream"))

print("Clickstream ingestion stream is running in the background...")


In [0]:
spark.streams.active

##Phase 3: The Bronze Layer – Raw Ingestion

In [0]:
%sql
--select * from dd_hr_sandpit.ecommerce_project.bronze_customers limit 10;
--select * from dd_hr_sandpit.ecommerce_project.bronze_products limit 10;
select * from dd_hr_sandpit.ecommerce_project.bronze_clickstream limit 10